In [1]:
import json
import re
import random
from pathlib import Path
from datasets import load_dataset

TRAIN_IN   = Path("train.jsonl")
EVAL_IN    = Path("eval.jsonl")
TRAIN_OUT  = Path("train_merged.jsonl")
EVAL_OUT   = Path("eval_merged.jsonl")

CROWNELIUS_DATASET = "Crownelius/Creative-Writing-High-Quality-1300x"
CROWNELIUS_SPLIT   = "train"
MIN_WORDS  = 600
MAX_WORDS  = 2500

EVAL_RATIO = 0.05   # 5% of Crownelius goes to eval

SEED = 42
random.seed(SEED)

print("Config ready.")

Config ready.


In [2]:
def load_jsonl(path: Path) -> list[dict]:
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train_existing = load_jsonl(TRAIN_IN)
eval_existing  = load_jsonl(EVAL_IN)

print(f"Colleague's train rows : {len(train_existing)}")
print(f"Colleague's eval rows  : {len(eval_existing)}")
print(f"\nSample train keys      : {list(train_existing[0].keys())}")
print(f"Sample meta keys       : {list(train_existing[0]['meta'].keys())}")
print(f"\nWord count range (train): "
      f"{min(r['meta']['word_count'] for r in train_existing)} – "
      f"{max(r['meta']['word_count'] for r in train_existing)}")

Colleague's train rows : 5219
Colleague's eval rows  : 274

Sample train keys      : ['instruction', 'output', 'source', 'meta']
Sample meta keys       : ['raw_prompt', 'word_count', 'score']

Word count range (train): 600 – 2475


In [3]:
raw_crownelius = load_dataset(CROWNELIUS_DATASET, split=CROWNELIUS_SPLIT)
print(f"Raw Crownelius rows: {len(raw_crownelius)}")
print(f"Columns: {raw_crownelius.column_names}")
print(f"\nSample row keys: {list(raw_crownelius[0].keys())}")
print(f"\n--- instruction sample ---")
print(raw_crownelius[0]["instruction"][:300])
print(f"\n--- output sample (first 500 chars) ---")
print(raw_crownelius[0]["output"][:500])

README.md: 0.00B [00:00, ?B/s]

c:\Users\Chis Bogdan-Mihai\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Chis Bogdan-Mihai\.cache\huggingface\hub\datasets--Crownelius--Creative-Writing-High-Quality-1300x. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but

train.jsonl:   0%|          | 0.00/16.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1386 [00:00<?, ? examples/s]

Raw Crownelius rows: 1386
Columns: ['instruction', 'input', 'output', 'system', 'metadata']

Sample row keys: ['instruction', 'input', 'output', 'system', 'metadata']

--- instruction sample ---
Write a dystopian scene where two characters engage in escaping a collapsing cave. You must track the physical location of every limb and the transfer of weight. Avoid metaphors; use precise anatomical and geometric descriptions. Constraint: The lighting is strobing or intermittent.

--- output sample (first 500 chars) ---
**STAGE BLOCKING:**
**STAGE BLOCKING SKELETON: DYSTOPIAN CAVE COLLAPSE**

**ENVIRONMENT & CONSTRAINT:**
*   **Coordinate System:** X-axis (left/right along cave passage), Y-axis (vertical, height), Z-axis (depth into/out of cave). Origin (0,0,0) at cave mouth.
*   **Lighting:** Intermittent strobing source (flickering bioluminescent fungi or failing artificial light). Duration: 0.3s light / 1.2s darkness per cycle. Action is segmented by these flashes.
*   **Structure:** Narr

In [4]:
def strip_stage_blocking(text: str) -> str:
    """
    Crownelius outputs begin with a structured 'Stage Blocking' skeleton
    before the actual prose. This function strips everything up to and
    including that skeleton, returning only the prose.

    The skeleton always ends with a markdown separator or a heading like
    '## Prose', '---', or '**Story:**' before the actual story begins.
    """
    # Common delimiters that separate skeleton from prose in Crownelius
    patterns = [
        r"(?:^|\n)#{1,3}\s*(?:prose|story|narrative|the story|writing)\s*\n",
        r"\n---+\n(?!.*\n---+\n)",   # last horizontal rule
        r"\*\*(?:prose|story|the story|narrative)\*\*[:\s]*\n",
        r"\n\n(?=(?:[A-Z\"][^\n]{20,}))",  # fallback: large gap before prose-like start
    ]

    for pattern in patterns:
        match = re.search(pattern, text, flags=re.IGNORECASE)
        if match:
            prose = text[match.end():].strip()
            if len(prose.split()) >= MIN_WORDS:
                return prose

    # If no delimiter found, return full text (may already be prose-only)
    return text.strip()


def count_words(text: str) -> int:
    return len(text.split())


# Test on a few rows
for i in range(3):
    raw_out = raw_crownelius[i]["output"]
    stripped = strip_stage_blocking(raw_out)
    wc_before = count_words(raw_out)
    wc_after  = count_words(stripped)
    print(f"Row {i}: {wc_before} words → {wc_after} words after stripping")
    print(f"  First 200 chars of prose: {stripped[:200]!r}")
    print()

Row 0: 1463 words → 675 words after stripping
  First 200 chars of prose: 'Character B was at X=7m, Y=1.5m, Z=2.5m. Crouched directly behind A. Both hands gripped the straps of A’s pack; arms fully extended, pulling his upper body forward. Left knee pressed into the small of'

Row 1: 1734 words → 836 words after stripping
  First 200 chars of prose: 'They began the standard waltz box step, the 1-2-3 count a metronome beneath the hiss of hydraulics and the scrape of hobnails. Character A initiated with her left foot. The metatarsals planted at (1.8'

Row 2: 1988 words → 1065 words after stripping
  First 200 chars of prose: 'The grid is absolute. Ground level is z=0. Zone 1 is a 10m x 15m rectangle at z=12m. Character B is 3m ahead of Character A, both on this rooftop. A’s initial position is (x=0, y=0). B is at (x=4, y=1'



In [5]:
def crownelius_to_schema(row: dict) -> dict | None:
    """
    Converts a raw Crownelius row to the unified training schema:
      { instruction, output, source, meta }

    Returns None if the row fails quality filters.
    """
    prose = strip_stage_blocking(row["output"])
    wc = count_words(prose)

    # Word count filter
    if not (MIN_WORDS <= wc <= MAX_WORDS):
        return None

    # Skip if stripping failed badly (skeleton still present)
    # Heuristic: Stage Blocking headers contain bullet points or numbered lists
    skeleton_indicators = re.findall(r"^\s*[-\d]+[\.\)]\s+\w", prose, flags=re.MULTILINE)
    if len(skeleton_indicators) > 5:
        return None

    instruction = row["instruction"].strip()

    return {
        "instruction": instruction,
        "output":      prose,
        "source":      "crownelius",
        "meta": {
            "raw_prompt": instruction,
            "word_count": wc,
        }
    }


crownelius_converted = []
skipped = 0

for row in raw_crownelius:
    converted = crownelius_to_schema(row)
    if converted:
        crownelius_converted.append(converted)
    else:
        skipped += 1

print(f"Crownelius rows converted : {len(crownelius_converted)}")
print(f"Crownelius rows skipped   : {skipped}")

wc_list = [r["meta"]["word_count"] for r in crownelius_converted]
print(f"Word count range          : {min(wc_list)} – {max(wc_list)}")
print(f"Average word count        : {sum(wc_list) // len(wc_list)}")

Crownelius rows converted : 1372
Crownelius rows skipped   : 14
Word count range          : 602 – 2002
Average word count        : 1030


In [6]:
random.shuffle(crownelius_converted)

n_eval  = max(1, int(len(crownelius_converted) * EVAL_RATIO))
n_train = len(crownelius_converted) - n_eval

crownelius_train = crownelius_converted[:n_train]
crownelius_eval  = crownelius_converted[n_train:]

# Merge
train_merged = train_existing + crownelius_train
eval_merged  = eval_existing  + crownelius_eval

# Shuffle merged sets so Crownelius rows aren't all at the end
random.shuffle(train_merged)
random.shuffle(eval_merged)

print(f"=== Final dataset sizes ===")
print(f"Train : {len(train_existing):>5} (colleague) + {len(crownelius_train):>4} (crownelius) = {len(train_merged):>5}")
print(f"Eval  : {len(eval_existing):>5} (colleague) + {len(crownelius_eval):>4} (crownelius) = {len(eval_merged):>5}")

# Source breakdown
for split_name, split_data in [("Train", train_merged), ("Eval", eval_merged)]:
    counts = {}
    for r in split_data:
        counts[r["source"]] = counts.get(r["source"], 0) + 1
    print(f"\n{split_name} sources: {counts}")

=== Final dataset sizes ===
Train :  5219 (colleague) + 1304 (crownelius) =  6523
Eval  :   274 (colleague) +   68 (crownelius) =   342

Train sources: {'writingprompts': 3783, 'gpt4o_writingprompts': 1436, 'crownelius': 1304}

Eval sources: {'crownelius': 68, 'gpt4o_writingprompts': 64, 'writingprompts': 210}


In [7]:
def save_jsonl(data: list[dict], path: Path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for row in data:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

save_jsonl(train_merged, TRAIN_OUT)
save_jsonl(eval_merged,  EVAL_OUT)

print(f"Saved: {TRAIN_OUT}  ({TRAIN_OUT.stat().st_size / 1e6:.1f} MB)")
print(f"Saved: {EVAL_OUT}   ({EVAL_OUT.stat().st_size / 1e6:.1f} MB)")

Saved: train_merged.jsonl  (42.7 MB)
Saved: eval_merged.jsonl   (2.2 MB)


In [8]:
# Re-load and verify nothing got corrupted
train_check = load_jsonl(TRAIN_OUT)
eval_check  = load_jsonl(EVAL_OUT)

assert len(train_check) == len(train_merged), "Train row count mismatch"
assert len(eval_check)  == len(eval_merged),  "Eval row count mismatch"

for r in train_check[:5]:
    assert "instruction" in r and "output" in r, f"Missing fields in row: {r.keys()}"
    assert len(r["output"].split()) >= MIN_WORDS,  f"Output too short: {len(r['output'].split())} words"

# Final distribution
all_wc = [r["meta"]["word_count"] for r in train_check]
buckets = {"600-800": 0, "800-1200": 0, "1200-1800": 0, "1800-2500": 0}
for wc in all_wc:
    if   wc <= 800:  buckets["600-800"]   += 1
    elif wc <= 1200: buckets["800-1200"]  += 1
    elif wc <= 1800: buckets["1200-1800"] += 1
    else:            buckets["1800-2500"] += 1

print("All assertions passed.\n")
print("=== Word count distribution (train) ===")
total = len(all_wc)
for bucket, count in buckets.items():
    bar = "█" * (count * 40 // total)
    print(f"  {bucket:>12}  {bar:<40}  {count:>4} ({count*100//total}%)")

All assertions passed.

=== Word count distribution (train) ===
       600-800  ██████████                                1725 (26%)
      800-1200  ████████████████████                      3408 (52%)
     1200-1800  ████████                                  1338 (20%)
     1800-2500                                              52 (0%)
